# Projet 1 (CS-433) — Data preprocessing

This notebook prepares the data x_train and y_train, and it produces **2 preprocessed versions** :

| Version | Use `y` ? | What it does |
|---|---|---|
| **v1 — X only** | no | cleanup, special blocks, NaN, outliers, normalisation, redundant columns |
| **v2 — X + y** | yes | same as v1, **plus** selection of features based on correlation with the target |

I only used the standard library, numpy and matplotlib/seaborn to respect the project guidelines

I print and save the generated figures on the repository

The preprocessing code itself (`fit_preprocessor`, `transform`, ...) lives in `src/preprocessing.py`, so that `run.py` can reuse it. This notebook imports it, applies it, plots the results and saves the files.


# **HOW TO USE**

The preprocessing has already been run, so you normally do not need to rerun this notebook.
Ready-to-use datasets are available in data/processed/

## **Which file should I use?**

train_v1_X_only.npz → recommended default

train_v2_X_y.npz → same dataset with additional target-based feature selection

train_v2_X_y_balanced.npz → balanced version for quick experiments only

test_v1_X_only.npz / test_v2_X_y.npz → matching test sets for final predictions

## **Always use matching train/test versions:**

train_v1_X_only.npz  <->  test_v1_X_only.npz
train_v2_X_y.npz     <->  test_v2_X_y.npz

## **Loading the data**

```python
import numpy as np

# Training data
d = np.load("data/processed/train_v1_X_only.npz")

x = d["x"]
y = d["y"]
ids = d["ids"]
feature_names = d["feature_names"]

# Add bias column expected by the project functions
tx = np.c_[np.ones(len(x)), x]

# Matching test data
t = np.load("data/processed/test_v1_X_only.npz")

x_test = t["x"]
ids_test = t["ids"]
tx_test = np.c_[np.ones(len(x_test)), x_test]
```

## **Important conventions**

y is encoded as {0, 1} for training.

AIcrowd submissions require predictions in {-1, +1}:

```python
y_pred_submission = 2 * y_pred - 1
```

Features are already imputed and standardized.

There are no remaining NaN values.

Arrays are stored as float32; convert to float64 if needed:

```python
x = x.astype(np.float64)
```

For most experiments, start with train_v1_X_only.npz.
The balanced dataset should only be used for quick training experiments, not for validation or evaluation.

## **Using the preprocessing inside run.py**

The code is in `src/preprocessing.py` (numpy only). `helpers.load_csv_data` drops the `Id` column and
does not return the column names, so use `load_feature_names` to get them. Fit on the training rows only,
then reuse the same parameters on validation and test rows:

```python
from src.preprocessing import fit_preprocessor, load_feature_names, transform

header = load_feature_names(DATA / "x_train.csv")
params, z_train = fit_preprocessor(x_train[tr], header, dtype=np.float64)
z_val = transform(x_train[val], header, params, dtype=np.float64)
z_test = transform(x_test, header, params, dtype=np.float64)
tx_train = np.c_[np.ones(len(z_train)), z_train]  # add the bias column
```

`select_by_target_corr` and `undersample_majority` (same file) use the labels: call them on the training part of each fold only.


## Imports and parameters

In [ ]:
import json
import sys
import time
from collections import defaultdict
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

SEED = 42  # for reproducibility

NAN_COL_MAX = 0.50  # delete col with more than 50% of Nan
CAT_MAX_UNIQUE = 15  # <= 15 distinct values -> discrete variable otherwise continuous
CLIP_QUANTILES = (0.005, 0.995)  # limits outliers
CORR_X_MAX = 0.95  # if |corr| > 0.95 between two cols, keep only the first

# --- v2 only (use y_train) ---
CORR_Y_MIN = 0.01  # keep features with |corr(feature, y)| >= 0.01
BALANCE_RATIO = 1.0

# --- options ---
SAVE_CSV = False  # True : exports also .csv
PROCESS_TEST = True  # True : transforme also x_test.csv if present in data/

# graphics style
BLUE, ORANGE, GREY = "#2f6db5", "#d9822b", "#8a8f98"
sns.set_theme(
    style="whitegrid",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": GREY,
        "grid.color": "#e6e8eb",
        "figure.dpi": 110,
    },
)
rng = np.random.default_rng(SEED)

## Loading data

In [ ]:
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "x_train.csv").exists():
            return p
    raise FileNotFoundError(
        "data/x_train.csv introuvable : ouvre le notebook depuis le dossier du projet "
        "(projects/project1)."
    )


def load_csv_fast(path, chunk_lines=20000):
    """Load a CSV -> (column names, float64 array). Empty fields become NaN."""
    parts = []
    with open(path) as f:
        header = f.readline().strip().split(",")
        n = len(header)
        while True:
            lines = list(islice(f, chunk_lines))
            if not lines:
                break
            txt = "".join(lines).replace("\r", "")
            txt = (
                txt.replace(",,", ",nan,")
                .replace(",,", ",nan,")
                .replace(",\n", ",nan\n")
                .replace("\n,", "\nnan,")
            )
            tokens = txt.replace("\n", ",").rstrip(",").split(",")
            parts.append(np.array(tokens, dtype=np.float64).reshape(-1, n))
    return header, np.vstack(parts)


ROOT = find_project_root()
DATA = ROOT / "data"
OUT = DATA / "processed"
OUT.mkdir(exist_ok=True)

# where figures go after being saves
FIG_DIR = ROOT / "figures" / "preprocessing"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name):
    """Saves fig for overleaf pdf"""
    for ext in ("pdf", "png"):
        plt.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"figure sauvegardée : {(FIG_DIR / name).relative_to(ROOT)}.pdf / .png")


print("Racine du projet :", ROOT)

t0 = time.time()
names, X_raw = load_csv_fast(DATA / "x_train.csv")
y_names, Y_raw = load_csv_fast(DATA / "y_train.csv")
print(f"x_train : {X_raw.shape}   y_train : {Y_raw.shape}   ({time.time() - t0:.0f} s)")


assert names[0] == "Id" and y_names == ["Id", "_MICHD"]
assert np.array_equal(X_raw[:, 0], Y_raw[:, 0]), (
    "Les Id de x_train et y_train ne sont pas alignés !"
)
ids = X_raw[:, 0].astype(np.int64)
y_pm1 = Y_raw[:, 1]  # étiquettes originales : -1 / +1
assert set(np.unique(y_pm1)) == {-1.0, 1.0}
y = (y_pm1 == 1).astype(
    np.int8
)  # étiquettes en {0, 1} (requis par la régression logistique du projet)
print("Étiquettes converties de {-1, +1} vers {0, 1}.")

In [ ]:
# The preprocessing code is shared with run.py: it lives in src/preprocessing.py
sys.path.insert(0, str(ROOT / "src"))
from preprocessing import (
    DROP_ADMIN,
    DROP_MIXED_UNITS,
    EXPLICIT_CODES,
    duplicate_row_mask,
    fit_preprocessor,
    select_by_target_corr,
    target_correlation,
    transform,
    undersample_majority,
)

## Raw data

In [ ]:
n, d = X_raw.shape[0], X_raw.shape[1] - 1
n_pos = int(y.sum())
n_neg = n - n_pos
print(
    f"{n} individus, {d} features. Positifs (MICHD=1) : {n_pos} ({n_pos / n:.1%}), "
    f"négatifs : {n_neg} ({n_neg / n:.1%})"
)

fig, ax = plt.subplots(figsize=(4.6, 3.2))
bars = ax.bar(
    ["0 : pas de MICHD", "1 : MICHD"], [n_neg, n_pos], color=[BLUE, ORANGE], width=0.55
)
for b, v in [(bars[0], n_neg), (bars[1], n_pos)]:
    ax.text(
        b.get_x() + b.get_width() / 2,
        v,
        f"{v:,}\n({v / n:.1%})".replace(",", " "),
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylim(0, n_neg * 1.18)
ax.set_ylabel("nombre d'individus")
ax.set_title("La cible est très déséquilibrée")
plt.tight_layout()
save_fig("01_class_balance")
plt.show()

In [ ]:
nan_frac = np.isnan(X_raw[:, 1:]).mean(axis=0)
rows_with_nan = np.isnan(X_raw[:, 1:]).any(axis=1).mean()
print(
    f"Lignes contenant au moins un NaN : {rows_with_nan:.1%}   <- retirer ces lignes "
    "viderait tout le dataset !"
)
print(
    f"Colonnes avec >{NAN_COL_MAX:.0%} de NaN : {(nan_frac > NAN_COL_MAX).sum()} sur "
    f"{d}   |   sans aucun NaN : {(nan_frac == 0).sum()}"
)

fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.hist(nan_frac, bins=20, color=BLUE, edgecolor="white")
ax.axvline(NAN_COL_MAX, color=ORANGE, lw=2)
ax.text(
    NAN_COL_MAX - 0.01,
    ax.get_ylim()[1] * 0.92,
    "seuil de suppression",
    color="#333",
    ha="right",
    fontsize=9,
)
ax.set_xlabel("fraction de NaN dans la colonne")
ax.set_ylabel("nombre de colonnes")
ax.set_title("Beaucoup de colonnes sont presque vides")
plt.tight_layout()
save_fig("02_missing_values")
plt.show()

## Duplicates

In [ ]:
dup = duplicate_row_mask(X_raw[:, 1:])
print(f"Lignes dupliquées : {dup.sum()}")
if dup.any():
    X_raw, y, ids = X_raw[~dup], y[~dup], ids[~dup]
    print(f"Doublons retirés -> {len(X_raw)} lignes restantes")

## Cols to remove because special code BRFSS

**Some cols aren't real features**
- identifiers, dates etc.(`SEQNO`, `_PSU`, `IDATE`, …) and nominal codes without order (`_STATE`, `EXRACT11`, `EXRACT21`)
- cols « raw » mixing multiple units (ex. `ALCDAY5` : 1xx = days per week, 2xx = days per month ; `WEIGHT2` : pounds or kg). Keep only the variables that summarize these values (`_DRNKWEK`, `WTKG3`, `HTM4`, `FTJUDA1_`, …)

(The lists of columns and codes are defined at the top of `src/preprocessing.py`.)

**Special BRFSS code** : `7`/`9` (don't know / refuse), `77`/`99`, `777`/`999`… and `88`

In [ ]:
# The lists and special codes are defined in src/preprocessing.py
print("Administrative / nominal columns dropped:", DROP_ADMIN)
print("Mixed-unit raw columns dropped:", DROP_MIXED_UNITS)
print("Columns with explicit special codes:", list(EXPLICIT_CODES))

missing_in_header = [
    c for c in DROP_ADMIN + DROP_MIXED_UNITS + list(EXPLICIT_CODES) if c not in names
]
print("Colonnes de ces listes absentes du fichier :", missing_in_header or "aucune")

## Preprocessor

For each remaining col `fit_preprocessor`does in this order:

1. special codes → NaN (or 0 for "none")
2. deletion if too many NaN or if the col is constant
3. outliers : for continuous variables → *winsoriation* to quantiles `CLIP_QUANTILES`
4. imputation of NaN
5. normalisation (*z-score*) → good for methods with gradient descent!
6. deletion of redundant cols

The code is in `src/preprocessing.py`.

`transform` reapplies these steps with the saved params at the beginning of the notebook -> we will use this on x_test

In [ ]:
cfg = {
    "nan_col_max": NAN_COL_MAX,
    "cat_max_unique": CAT_MAX_UNIQUE,
    "clip_quantiles": list(CLIP_QUANTILES),
    "corr_x_max": CORR_X_MAX,
}
t0 = time.time()
params_v1, Z_fit = fit_preprocessor(X_raw, names, cfg)
Z = transform(X_raw, names, params_v1)
assert np.allclose(Z, Z_fit, atol=1e-5), (
    "fit et transform ne donnent pas le même résultat !"
)
feature_names = np.array([c["name"] for c in params_v1["columns"]])
print(f"Terminé en {time.time() - t0:.0f} s  ->  Z : {Z.shape}")

## What has been deleted

In [ ]:
print(f"Colonnes de départ : {d}   ->   colonnes gardées : {Z.shape[1]}\n")
for reason, lst in params_v1["dropped"].items():
    print(f"- {len(lst):3d} colonnes : {reason}")
    if len(lst) <= 12:
        print("      ", ", ".join(lst))

groups = defaultdict(list)
for c in params_v1["columns"]:
    if c["nan_codes"] or c["zero_codes"]:
        groups[(tuple(c["nan_codes"]), tuple(c["zero_codes"]))].append(c["name"])
print(
    "\nColonnes gardées avec codes spéciaux traités : "
    f"{sum(map(len, groups.values()))}  (à vérifier avec le codebook)"
)
for (a, b), lst in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    print(
        f"\n  NaN pour {list(a) or '-'}"
        + (f", 0 pour {list(b)}" if b else "")
        + f"  ({len(lst)} colonnes) :"
    )
    print("    " + ", ".join(lst))
n_cont = sum(c["kind"] == "cont" for c in params_v1["columns"])
print(
    f"\nTypes des colonnes gardées : {n_cont} continues, {Z.shape[1] - n_cont} "
    "catégorielles / discrètes"
)

## Quality control

In [ ]:
assert not np.isnan(Z).any() and not np.isinf(Z).any(), "NaN ou inf restants"
assert np.allclose(Z.mean(axis=0, dtype=np.float64), 0, atol=1e-3) and np.allclose(
    Z.std(axis=0, dtype=np.float64), 1, atol=1e-3
), "normalisation incorrecte"
assert Z.shape[0] == len(y) == len(ids)
print(
    "OK : plus aucun NaN/inf, moyenne 0 et écart-type 1 pour chaque colonne, "
    "dimensions cohérentes."
)

## Before/after on a couple of variables

In [ ]:
demo = [f for f in ["PHYSHLTH", "_BMI5", "_AGE80"] if f in feature_names]
sample = rng.choice(len(X_raw), size=50_000, replace=False)
col_of = {nm: i for i, nm in enumerate(names)}
fig, axes = plt.subplots(2, len(demo), figsize=(3.6 * len(demo), 5), squeeze=False)
for k, f in enumerate(demo):
    raw = X_raw[sample, col_of[f]]
    raw = raw[~np.isnan(raw)]
    axes[0, k].hist(raw, bins=40, color=GREY, edgecolor="white")
    axes[0, k].set_title(f"{f} — brut", fontsize=10)
    axes[1, k].hist(
        Z[sample, list(feature_names).index(f)], bins=40, color=BLUE, edgecolor="white"
    )
    axes[1, k].set_title(f"{f} — traité", fontsize=10)
axes[0, 0].set_ylabel("brut")
axes[1, 0].set_ylabel("traité")
fig.suptitle(
    "Effet du pré-traitement (ex. PHYSHLTH : 88 = « aucun jour » devient 0)",
    y=1.0,
    fontsize=11,
)
plt.tight_layout()
save_fig("03_before_after")
plt.show()

## Saving version 1 (x_train alone)

In [ ]:
def save_npz(path, **arrays):
    np.savez(path, **arrays)
    print(f"  -> {path.relative_to(ROOT)}  ({path.stat().st_size / 1e6:.0f} Mo)")


def save_params(path, params, **extra):
    with open(path, "w") as f:
        json.dump({**params, **extra}, f, indent=1)
    print(f"  -> {path.relative_to(ROOT)}")


def save_csv(path, x, y, ids, feature_names):
    """Export CSV optionnel : Id, _MICHD, puis les features (lent : ~1-2 min)."""
    mat = np.column_stack([ids, y, x])
    np.savetxt(
        path,
        mat,
        delimiter=",",
        header=",".join(["Id", "_MICHD", *feature_names]),
        comments="",
        fmt="%.6g",
    )
    print(f"  -> {path.relative_to(ROOT)}")


print("Version 1 (X seul) :")
save_npz(OUT / "train_v1_X_only.npz", x=Z, y=y, ids=ids, feature_names=feature_names)
save_params(OUT / "preprocess_params_v1.json", params_v1)
if SAVE_CSV:
    save_csv(OUT / "train_v1_X_only.csv", Z, y, ids, feature_names)

## Saving version 2 (preprocessing on x_train **and** y_train)

In [ ]:
r = target_correlation(Z, y)
order = np.argsort(-np.abs(r))
fig, ax = plt.subplots(figsize=(6.4, 4.6))
top = order[:20][::-1]
ax.barh(feature_names[top], r[top], color=[BLUE if v > 0 else ORANGE for v in r[top]])
ax.axvline(0, color=GREY, lw=1)
ax.set_xlabel("corrélation avec y (bleu : positive, orange : négative)")
ax.set_title("Les 20 features les plus corrélées à MICHD")
plt.tight_layout()
save_fig("04_target_correlation")
plt.show()
print(
    "Attention : le signe dépend du codage BRFSS (ex. pour beaucoup de questions 1 = "
    "oui, 2 = non, donc une corrélation négative "
    "signifie « répondre oui va avec MICHD = 1 »). Seule la valeur absolue compte "
    "pour la sélection."
)

sel = select_by_target_corr(Z, y, CORR_Y_MIN)
print(f"Features avec |corr| >= {CORR_Y_MIN} : {len(sel)} sur {Z.shape[1]}")

In [ ]:
# Heatmap des 15 features les plus corrélées à y (redondances éventuelles)
top15 = order[:15]
Cm = np.corrcoef(Z[:, top15].T)
fig, ax = plt.subplots(figsize=(7, 5.8))
sns.heatmap(
    Cm,
    xticklabels=feature_names[top15],
    yticklabels=feature_names[top15],
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("Corrélations entre les 15 features les plus liées à y")
plt.tight_layout()
save_fig("05_correlation_heatmap")
plt.show()

In [ ]:
# Application : v2 = mêmes paramètres que v1, restreints aux features sélectionnées
params_v2 = dict(params_v1, columns=[params_v1["columns"][i] for i in sel])
Z2 = Z[:, sel]
feature_names_v2 = feature_names[sel]

bal_idx = undersample_majority(y, BALANCE_RATIO, rng)
Z2_bal, y_bal, ids_bal = Z2[bal_idx], y[bal_idx], ids[bal_idx]
print(
    f"v2 (toutes les lignes) : {Z2.shape}   |   v2 rééquilibrée : {Z2_bal.shape}, "
    f"positifs = {y_bal.mean():.1%}"
)

print("Version 2 (X + y) :")
save_npz(OUT / "train_v2_X_y.npz", x=Z2, y=y, ids=ids, feature_names=feature_names_v2)
save_npz(
    OUT / "train_v2_X_y_balanced.npz",
    x=Z2_bal,
    y=y_bal,
    ids=ids_bal,
    feature_names=feature_names_v2,
)
save_params(
    OUT / "preprocess_params_v2.json",
    params_v2,
    target_corr_threshold=CORR_Y_MIN,
    balance_ratio=BALANCE_RATIO,
    seed=SEED,
)
if SAVE_CSV:
    save_csv(OUT / "train_v2_X_y.csv", Z2, y, ids, feature_names_v2)
    save_csv(
        OUT / "train_v2_X_y_balanced.csv", Z2_bal, y_bal, ids_bal, feature_names_v2
    )

## Apply same preprocessing to x_test

In [ ]:
test_path = DATA / "x_test.csv"
if PROCESS_TEST and test_path.exists():
    test_names, X_test_raw = load_csv_fast(test_path)
    assert test_names == names, (
        "Les colonnes de x_test ne correspondent pas à celles de x_train"
    )
    ids_test = X_test_raw[:, 0].astype(np.int64)
    Zt1 = transform(X_test_raw, test_names, params_v1)
    Zt2 = transform(X_test_raw, test_names, params_v2)
    assert not np.isnan(Zt1).any() and not np.isnan(Zt2).any()
    print("Test :")
    save_npz(
        OUT / "test_v1_X_only.npz", x=Zt1, ids=ids_test, feature_names=feature_names
    )
    save_npz(
        OUT / "test_v2_X_y.npz", x=Zt2, ids=ids_test, feature_names=feature_names_v2
    )
else:
    print(
        "x_test.csv absent de data/ (ou PROCESS_TEST=False) : étape ignorée. Relance "
        "ce notebook quand il sera ajouté."
    )